In [ ]:
!pip install -q sentence-transformers datasets ir-datasets pandas numpy matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.9 MB/s eta 0:00:00


In [ ]:
import ir_datasets

dataset = ir_datasets.load("beir/nfcorpus/test")

print(dataset)

Dataset(id='beir/nfcorpus/test', provides=['docs', 'queries', 'qrels'])


In [ ]:
docs = list(dataset.docs_iter())

print("Number of documents:", len(docs))
print(docs[0])

Number of documents: 3633
BeirTitleUrlDoc(doc_id='MED-10', text='Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of 3.25 years after the diagnosis (range 0.08–9.0 years) 6,011 participants died, of which 3,619 (60.2%) was 

[INFO] Opening /root/.ir_datasets/beir/nfcorpus/docs.pklz4/bin with direct file access


In [ ]:
queries = list(dataset.queries_iter())

print("Number of queries:", len(queries))
print(queries[0])

Number of queries: 323
BeirUrlQuery(query_id='PLAIN-2', text='Do Cholesterol Statin Drugs Cause Breast Cancer?', url='http://nutritionfacts.org/2015/07/16/do-cholesterol-statin-drugs-cause-breast-cancer/')


[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]
[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]


In [ ]:
qrels = list(dataset.qrels_iter())

print("Number of relevance judgments:", len(qrels))
print(qrels[0])

Number of relevance judgments: 12334
TrecQrel(query_id='PLAIN-2', doc_id='MED-2427', relevance=2, iteration='0')


In [ ]:
print("Documents:", len(docs))
print("Queries:", len(queries))
print("Qrels:", len(qrels))

Documents: 3633
Queries: 323
Qrels: 12334


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("SBERT model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT model loaded successfully!


In [ ]:
test_sentence = "This is a test sentence."

embedding = model.encode(test_sentence)

print("Embedding shape:", embedding.shape)

Embedding shape: (384,)


In [ ]:
doc_ids = [doc.doc_id for doc in docs]
doc_texts = [doc.title + " " + doc.text for doc in docs]

query_ids = [query.query_id for query in queries]
query_texts = [query.text for query in queries]

print("Documents:", len(doc_texts))
print("Queries:", len(query_texts))

Documents: 3633
Queries: 323


In [ ]:
import time

start = time.time()

doc_embeddings = model.encode(
    doc_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

doc_encoding_time = time.time() - start

print("Document embeddings:", doc_embeddings.shape)
print("Encoding time:", round(doc_encoding_time, 2), "seconds")

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

Document embeddings: torch.Size([3633, 384])
Encoding time: 11.14 seconds


In [ ]:
start = time.time()

query_embeddings = model.encode(
    query_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

query_encoding_time = time.time() - start

print("Query embeddings:", query_embeddings.shape)
print("Query encoding time:", round(query_encoding_time, 2), "seconds")

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Query embeddings: torch.Size([323, 384])
Query encoding time: 0.29 seconds


In [ ]:
from sentence_transformers import util

start = time.time()

sbert_results = util.semantic_search(
    query_embeddings,
    doc_embeddings,
    top_k=100
)

retrieval_time = time.time() - start

print("Queries searched:", len(sbert_results))
print("Retrieval time:", round(retrieval_time, 2), "seconds")

Queries searched: 323
Retrieval time: 0.08 seconds


In [ ]:
q = 0

print("QUERY:")
print(query_texts[q])

print("\nTOP 5 SBERT RESULTS:")

for result in sbert_results[q][:5]:
    idx = result["corpus_id"]

    print("\nScore:", round(result["score"], 4))
    print("Document:", doc_texts[idx][:300])

QUERY:
Do Cholesterol Statin Drugs Cause Breast Cancer?

TOP 5 SBERT RESULTS:

Score: 0.6476
Document: Total Cholesterol and Cancer Risk in a Large Prospective Study in Korea Purpose To further clarify the relationship between total cholesterol and cancer, which remains unclear. Methods We prospectively examined the association between total cholesterol and site-specific and all-cancer incidence amon

Score: 0.6237
Document: Statin use and risk of breast cancer: a meta-analysis of observational studies. Emerging evidence suggests that statins' may decrease the risk of cancers. However, available evidence on breast cancer is conflicting. We, therefore, examined the association between statin use and risk of breast cancer

Score: 0.6208
Document: Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence bu

In [ ]:
from collections import defaultdict
relevant_docs = defaultdict(dict)

for qrel in qrels:
    relevant_docs[qrel.query_id][qrel.doc_id] = qrel.relevance

print("Queries with relevance judgments:", len(relevant_docs))
first_qid = query_ids[0]
print("Query ID:", first_qid)
print("Relevant documents:", relevant_docs[first_qid])

Queries with relevance judgments: 323
Query ID: PLAIN-2
Relevant documents: {'MED-2427': 2, 'MED-10': 2, 'MED-2429': 2, 'MED-2430': 2, 'MED-2431': 2, 'MED-14': 2, 'MED-2432': 2, 'MED-2428': 1, 'MED-2440': 1, 'MED-2434': 1, 'MED-2435': 1, 'MED-2436': 1, 'MED-2437': 1, 'MED-2438': 1, 'MED-2439': 1, 'MED-3597': 1, 'MED-3598': 1, 'MED-3599': 1, 'MED-4556': 1, 'MED-4559': 1, 'MED-4560': 1, 'MED-4828': 1, 'MED-4829': 1, 'MED-4830': 1}


In [ ]:
sbert_rankings = {}

for i, results in enumerate(sbert_results):
    qid = query_ids[i]

    ranked_docs = []

    for result in results:
        corpus_index = result["corpus_id"]
        doc_id = doc_ids[corpus_index]
        score = float(result["score"])

        ranked_docs.append((doc_id, score))

    sbert_rankings[qid] = ranked_docs

print("SBERT rankings created:", len(sbert_rankings))

SBERT rankings created: 323


In [ ]:
import math
import numpy as np

def dcg(relevances):
    score = 0.0

    for i, rel in enumerate(relevances):
        score += (2**rel - 1) / math.log2(i + 2)

    return score


def ndcg_at_k(qid, ranking, k=10):

    actual = relevant_docs[qid]

    retrieved_rels = [
        actual.get(doc_id, 0)
        for doc_id, _ in ranking[:k]
    ]

    ideal_rels = sorted(actual.values(), reverse=True)[:k]

    actual_dcg = dcg(retrieved_rels)
    ideal_dcg = dcg(ideal_rels)

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg

In [ ]:
ndcg_scores = []

for qid in query_ids:
    score = ndcg_at_k(
        qid,
        sbert_rankings[qid],
        k=10
    )

    ndcg_scores.append(score)

sbert_ndcg10 = np.mean(ndcg_scores)

print("SBERT nDCG@10:", round(sbert_ndcg10, 4))

SBERT nDCG@10: 0.3173


In [ ]:
def recall_at_k(qid, ranking, k=10):

    relevant = set(relevant_docs[qid].keys())

    retrieved = {
        doc_id
        for doc_id, _ in ranking[:k]
    }

    if len(relevant) == 0:
        return 0.0

    return len(relevant & retrieved) / len(relevant)


recall_scores = []

for qid in query_ids:

    score = recall_at_k(
        qid,
        sbert_rankings[qid],
        k=10
    )

    recall_scores.append(score)

sbert_recall10 = np.mean(recall_scores)

print("SBERT Recall@10:", round(sbert_recall10, 4))

SBERT Recall@10: 0.155


In [ ]:
def reciprocal_rank_at_k(qid, ranking, k=10):

    relevant = set(relevant_docs[qid].keys())

    for rank, (doc_id, _) in enumerate(ranking[:k], start=1):

        if doc_id in relevant:
            return 1.0 / rank

    return 0.0


mrr_scores = []

for qid in query_ids:

    score = reciprocal_rank_at_k(
        qid,
        sbert_rankings[qid],
        k=10
    )

    mrr_scores.append(score)

sbert_mrr10 = np.mean(mrr_scores)

print("SBERT MRR@10:", round(sbert_mrr10, 4))

SBERT MRR@10: 0.5077


In [ ]:
print("=== SBERT BASELINE RESULTS ===")
print("nDCG@10 :", round(sbert_ndcg10, 4))
print("Recall@10:", round(sbert_recall10, 4))
print("MRR@10  :", round(sbert_mrr10, 4))

print("\nTiming")
print("Document encoding:", round(doc_encoding_time, 2), "seconds")
print("Query encoding:", round(query_encoding_time, 2), "seconds")
print("Retrieval:", round(retrieval_time, 2), "seconds")

=== SBERT BASELINE RESULTS ===
nDCG@10 : 0.3173
Recall@10: 0.155
MRR@10  : 0.5077

Timing
Document encoding: 11.14 seconds
Query encoding: 0.29 seconds
Retrieval: 0.08 seconds


In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

print("Cross-Encoder loaded successfully!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder loaded successfully!


In [ ]:
test_pairs = [
    ["Do Cholesterol Statin Drugs Cause Breast Cancer?",
     "Statin use and risk of breast cancer: a meta-analysis of observational studies."],

    ["Do Cholesterol Statin Drugs Cause Breast Cancer?",
     "The effects of exercise on knee injuries."]
]

scores = cross_encoder.predict(test_pairs)

print(scores)

[  4.5752378 -11.324395 ]


In [ ]:
import time

def cross_encoder_rerank(k):

    reranked_results = {}
    start_time = time.time()

    for i, qid in enumerate(query_ids):

        query_text = query_texts[i]

        candidates = sbert_rankings[qid][:k]

        pairs = []
        candidate_ids = []

        for doc_id, sbert_score in candidates:

            doc_index = doc_ids.index(doc_id)
            document_text = doc_texts[doc_index]

            pairs.append([query_text, document_text])
            candidate_ids.append(doc_id)


        ce_scores = cross_encoder.predict(
            pairs,
            show_progress_bar=False
        )


        reranked = list(zip(candidate_ids, ce_scores))


        reranked.sort(
            key=lambda x: float(x[1]),
            reverse=True
        )

        reranked_results[qid] = reranked

        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(query_ids)} queries")

    total_time = time.time() - start_time

    return reranked_results, total_time

In [ ]:
ce_results_10, ce_time_10 = cross_encoder_rerank(10)

print("K=10 reranking completed!")
print("Runtime:", round(ce_time_10, 2), "seconds")

Processed 50/323 queries
Processed 100/323 queries
Processed 150/323 queries
Processed 200/323 queries
Processed 250/323 queries
Processed 300/323 queries
K=10 reranking completed!
Runtime: 19.4 seconds


In [ ]:
ce10_ndcg_scores = []
ce10_recall_scores = []
ce10_mrr_scores = []

for qid in query_ids:

    ranking = ce_results_10[qid]

    ce10_ndcg_scores.append(
        ndcg_at_k(qid, ranking, k=10)
    )

    ce10_recall_scores.append(
        recall_at_k(qid, ranking, k=10)
    )

    ce10_mrr_scores.append(
        reciprocal_rank_at_k(qid, ranking, k=10)
    )

ce10_ndcg10 = np.mean(ce10_ndcg_scores)
ce10_recall10 = np.mean(ce10_recall_scores)
ce10_mrr10 = np.mean(ce10_mrr_scores)

print("=== CROSS-ENCODER K=10 RESULTS ===")
print("nDCG@10 :", round(ce10_ndcg10, 4))
print("Recall@10:", round(ce10_recall10, 4))
print("MRR@10  :", round(ce10_mrr10, 4))
print("Reranking time:", round(ce_time_10, 2), "seconds")

=== CROSS-ENCODER K=10 RESULTS ===
nDCG@10 : 0.3312
Recall@10: 0.155
MRR@10  : 0.5425
Reranking time: 19.4 seconds


In [ ]:
ce_results_25, ce_time_25 = cross_encoder_rerank(25)

print("K=25 reranking completed!")
print("Runtime:", round(ce_time_25, 2), "seconds")

Processed 50/323 queries
Processed 100/323 queries
Processed 150/323 queries
Processed 200/323 queries
Processed 250/323 queries
Processed 300/323 queries
K=25 reranking completed!
Runtime: 51.13 seconds


In [ ]:
ce25_ndcg_scores = []
ce25_recall_scores = []
ce25_mrr_scores = []

for qid in query_ids:

    ranking = ce_results_25[qid]

    ce25_ndcg_scores.append(
        ndcg_at_k(qid, ranking, k=10)
    )

    ce25_recall_scores.append(
        recall_at_k(qid, ranking, k=10)
    )

    ce25_mrr_scores.append(
        reciprocal_rank_at_k(qid, ranking, k=10)
    )

ce25_ndcg10 = np.mean(ce25_ndcg_scores)
ce25_recall10 = np.mean(ce25_recall_scores)
ce25_mrr10 = np.mean(ce25_mrr_scores)

print("=== CROSS-ENCODER K=25 RESULTS ===")
print("nDCG@10 :", round(ce25_ndcg10, 4))
print("Recall@10:", round(ce25_recall10, 4))
print("MRR@10  :", round(ce25_mrr10, 4))
print("Reranking time:", round(ce_time_25, 2), "seconds")

=== CROSS-ENCODER K=25 RESULTS ===
nDCG@10 : 0.3376
Recall@10: 0.1524
MRR@10  : 0.5548
Reranking time: 51.13 seconds


In [ ]:
ce_results_50, ce_time_50 = cross_encoder_rerank(50)

ce50_ndcg10 = np.mean([
    ndcg_at_k(qid, ce_results_50[qid], 10)
    for qid in query_ids
])

ce50_recall10 = np.mean([
    recall_at_k(qid, ce_results_50[qid], 10)
    for qid in query_ids
])

ce50_mrr10 = np.mean([
    reciprocal_rank_at_k(qid, ce_results_50[qid], 10)
    for qid in query_ids
])

print("=== CROSS-ENCODER K=50 RESULTS ===")
print("nDCG@10 :", round(ce50_ndcg10, 4))
print("Recall@10:", round(ce50_recall10, 4))
print("MRR@10  :", round(ce50_mrr10, 4))
print("Reranking time:", round(ce_time_50, 2), "seconds")

Processed 50/323 queries
Processed 100/323 queries
Processed 150/323 queries
Processed 200/323 queries
Processed 250/323 queries
Processed 300/323 queries
=== CROSS-ENCODER K=50 RESULTS ===
nDCG@10 : 0.3413
Recall@10: 0.155
MRR@10  : 0.5686
Reranking time: 89.95 seconds


In [ ]:
ce_results_100, ce_time_100 = cross_encoder_rerank(100)

ce100_ndcg10 = np.mean([
    ndcg_at_k(qid, ce_results_100[qid], 10)
    for qid in query_ids
])

ce100_recall10 = np.mean([
    recall_at_k(qid, ce_results_100[qid], 10)
    for qid in query_ids
])

ce100_mrr10 = np.mean([
    reciprocal_rank_at_k(qid, ce_results_100[qid], 10)
    for qid in query_ids
])

print("=== CROSS-ENCODER K=100 RESULTS ===")
print("nDCG@10 :", round(ce100_ndcg10, 4))
print("Recall@10:", round(ce100_recall10, 4))
print("MRR@10  :", round(ce100_mrr10, 4))
print("Reranking time:", round(ce_time_100, 2), "seconds")

Processed 50/323 queries
Processed 100/323 queries
Processed 150/323 queries
Processed 200/323 queries
Processed 250/323 queries
Processed 300/323 queries
=== CROSS-ENCODER K=100 RESULTS ===
nDCG@10 : 0.3443
Recall@10: 0.1571
MRR@10  : 0.5706
Reranking time: 162.02 seconds


In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "System": [
        "SBERT",
        "SBERT + CE K=10",
        "SBERT + CE K=25",
        "SBERT + CE K=50",
        "SBERT + CE K=100"
    ],
    "nDCG@10": [
        sbert_ndcg10,
        ce10_ndcg10,
        ce25_ndcg10,
        ce50_ndcg10,
        ce100_ndcg10
    ],
    "Recall@10": [
        sbert_recall10,
        ce10_recall10,
        ce25_recall10,
        ce50_recall10,
        ce100_recall10
    ],
    "MRR@10": [
        sbert_mrr10,
        ce10_mrr10,
        ce25_mrr10,
        ce50_mrr10,
        ce100_mrr10
    ],
    "Reranking_Time_Seconds": [
        0,
        ce_time_10,
        ce_time_25,
        ce_time_50,
        ce_time_100
    ]
})

results_df.to_csv("results.csv", index=False)

print(results_df)
print("\nSaved as results.csv")

             System   nDCG@10  Recall@10    MRR@10  Reranking_Time_Seconds
0             SBERT  0.317265   0.154988  0.507654                0.000000
1   SBERT + CE K=10  0.331198   0.154988  0.542546               19.404914
2   SBERT + CE K=25  0.337601   0.152399  0.554755               51.130832
3   SBERT + CE K=50  0.341322   0.155033  0.568560               89.951049
4  SBERT + CE K=100  0.344284   0.157113  0.570647              162.019825

Saved as results.csv
